In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import re
from collections import Counter
from transformers import pipeline

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = [
    ("How can I reset my password?", "You can reset it through the account settings."),
    ("Where is my order?", "You can track your order in your profile."),
    ("How do I contact support?", "You can email us at support@example.com."),
    ("What are your business hours?", "We are open from 9 am to 5 pm."),
    ("Can I return an item?", "Yes, you have 30 days to return items."),
    ("How long does shipping take?", "Shipping usually takes 3 to 5 business days."),
    ("Do you offer international shipping?", "Yes, we ship worldwide."),
    ("Where are you located?", "Our main office is in New York."),
    ("How can I cancel my subscription?", "Go to billing and click cancel."),
    ("What payment methods do you accept?", "We accept credit cards and PayPal.")
]

def clean_text(text):
    return re.sub(r'[^\w\s]', '', text.lower())

questions = [clean_text(q).split() for q, _ in data]
answers = [clean_text(a).split() for _, a in data]

all_words = [word for seq in questions + answers for word in seq]
vocab_counts = Counter(all_words)

PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = '<PAD>', '<SOS>', '<EOS>', '<UNK>'
vocab = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2, UNK_TOKEN: 3}
for word in vocab_counts.keys():
    if word not in vocab:
        vocab[word] = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}

max_len_q = max(len(q) for q in questions)
max_len_a = max(len(a) for a in answers) + 1

print(f"Розмір словника (input_size / vocab_size): {len(vocab)}")
print(f"Макс. довжина входу: {max_len_q}, Макс. довжина виходу: {max_len_a}")

def process_data(questions, answers, vocab, max_len_q, max_len_a):
    enc_inputs, dec_inputs, dec_outputs = [], [], []
    for q, a in zip(questions, answers):
        q_idx = [vocab.get(w, vocab[UNK_TOKEN]) for w in q]
        q_idx += [vocab[PAD_TOKEN]] * (max_len_q - len(q_idx))

        a_idx = [vocab.get(w, vocab[UNK_TOKEN]) for w in a]
        dec_in = [vocab[SOS_TOKEN]] + a_idx
        dec_in += [vocab[PAD_TOKEN]] * (max_len_a - len(dec_in))

        dec_out = a_idx + [vocab[EOS_TOKEN]]
        dec_out += [vocab[PAD_TOKEN]] * (max_len_a - len(dec_out))
        
        enc_inputs.append(q_idx)
        dec_inputs.append(dec_in)
        dec_outputs.append(dec_out)
        
    return torch.tensor(enc_inputs).to(device), torch.tensor(dec_inputs).to(device), torch.tensor(dec_outputs).to(device)

X_enc, X_dec, Y_dec = process_data(questions, answers, vocab, max_len_q, max_len_a)
print("Дані успішно підготовлені у тензори.")

Розмір словника (input_size / vocab_size): 79
Макс. довжина входу: 6, Макс. довжина виходу: 10
Дані успішно підготовлені у тензори.


In [12]:
class Seq2SeqModel(nn.Module):
    def __init__(self, vocab_size, hidden_size=128, rnn_type='RNN'):
        super(Seq2SeqModel, self).__init__()
        self.rnn_type = rnn_type
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        
        if rnn_type == 'RNN':
            self.rnn = nn.RNN(input_size=hidden_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        else:
            self.rnn = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
            
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, enc_input, dec_input):
        embedded_enc = self.embedding(enc_input)

        if self.rnn_type == 'RNN':
            _, hidden = self.rnn(embedded_enc)
        else:
            _, (hidden, cell) = self.rnn(embedded_enc)

        embedded_dec = self.embedding(dec_input)
        if self.rnn_type == 'RNN':
            out, _ = self.rnn(embedded_dec, hidden)
        else:
            out, _ = self.rnn(embedded_dec, (hidden, cell))
            
        return self.fc(out)

vocab_size = len(vocab)
model_rnn = Seq2SeqModel(vocab_size, hidden_size=128, rnn_type='RNN').to(device)
criterion = nn.CrossEntropyLoss(ignore_index=vocab[PAD_TOKEN])
optimizer_rnn = optim.Adam(model_rnn.parameters(), lr=0.001)

print("Навчання RNN-моделі (10 епох)...")
for epoch in range(10):
    model_rnn.train()
    optimizer_rnn.zero_grad()
    
    output = model_rnn(X_enc, X_dec)
    loss = criterion(output.view(-1, vocab_size), Y_dec.view(-1))
    
    loss.backward()
    optimizer_rnn.step()
    print(f"Епоха [{epoch+1}/10], Втрати (Loss): {loss.item():.4f}")

Навчання RNN-моделі (10 епох)...
Епоха [1/10], Втрати (Loss): 4.3827
Епоха [2/10], Втрати (Loss): 4.2291
Епоха [3/10], Втрати (Loss): 4.0777
Епоха [4/10], Втрати (Loss): 3.9287
Епоха [5/10], Втрати (Loss): 3.7823
Епоха [6/10], Втрати (Loss): 3.6387
Епоха [7/10], Втрати (Loss): 3.4978
Епоха [8/10], Втрати (Loss): 3.3595
Епоха [9/10], Втрати (Loss): 3.2238
Епоха [10/10], Втрати (Loss): 3.0906


In [13]:
model_lstm = Seq2SeqModel(vocab_size, hidden_size=128, rnn_type='LSTM').to(device)
optimizer_lstm = optim.Adam(model_lstm.parameters(), lr=0.001)

print("Навчання LSTM-моделі (10 епох)...")
for epoch in range(10):
    model_lstm.train()
    optimizer_lstm.zero_grad()
    
    output = model_lstm(X_enc, X_dec)
    loss = criterion(output.view(-1, vocab_size), Y_dec.view(-1))
    
    loss.backward()
    optimizer_lstm.step()
    if (epoch + 1) % 2 == 0:
        print(f"Епоха [{epoch+1}/10], Втрати (Loss): {loss.item():.4f}")

def generate_response(model, question_text, max_len=10):
    model.eval()
    q_tokens = [vocab.get(w, vocab[UNK_TOKEN]) for w in clean_text(question_text).split()]
    enc_in = torch.tensor([q_tokens]).to(device)
    
    with torch.no_grad():
        emb_enc = model.embedding(enc_in)
        if model.rnn_type == 'RNN':
            _, hidden = model.rnn(emb_enc)
        else:
            _, (hidden, cell) = model.rnn(emb_enc)
            
        dec_in = torch.tensor([[vocab[SOS_TOKEN]]]).to(device)
        generated_words = []
        
        for _ in range(max_len):
            emb_dec = model.embedding(dec_in)
            if model.rnn_type == 'RNN':
                out, hidden = model.rnn(emb_dec, hidden)
            else:
                out, (hidden, cell) = model.rnn(emb_dec, (hidden, cell))
                
            out = model.fc(out)
            pred_token = out.argmax(dim=-1).item()
            
            if pred_token == vocab[EOS_TOKEN]:
                break
            generated_words.append(inv_vocab.get(pred_token, UNK_TOKEN))
            dec_in = torch.tensor([[pred_token]]).to(device)
            
    return " ".join(generated_words)

print("\n--- Генерація відповідей LSTM ---")
test_queries = [
    "How can I reset my password?",
    "Where is my order?",
    "Can I return an item?",
    "Do you offer international shipping?",
    "What payment methods do you accept?"
]

for q in test_queries:
    print(f"Q: {q}\nA: {generate_response(model_lstm, q)}\n")

Навчання LSTM-моделі (10 епох)...
Епоха [2/10], Втрати (Loss): 4.3154
Епоха [4/10], Втрати (Loss): 4.2115
Епоха [6/10], Втрати (Loss): 4.1044
Епоха [8/10], Втрати (Loss): 3.9903
Епоха [10/10], Втрати (Loss): 3.8660

--- Генерація відповідей LSTM ---
Q: How can I reset my password?
A: you can can email us at supportexamplecom

Q: Where is my order?
A: you can can email us at supportexamplecom

Q: Can I return an item?
A: you can email us at supportexamplecom

Q: Do you offer international shipping?
A: we accept credit cards and paypal

Q: What payment methods do you accept?
A: we accept credit cards and paypal



In [14]:
print("Завантаження моделі Transformer (distilGPT2)...")
generator = pipeline('text-generation', model='distilgpt2')

prompts = [
    "Customer: I forgot my password.\nSupport:", 
    "Customer: My order is delayed.\nSupport:"
]

print("\n--- Генерація відповідей Transformer ---")
for prompt in prompts:
    response = generator(prompt, max_new_tokens=20, num_return_sequences=1, temperature=0.7, pad_token_id=50256)[0]['generated_text']
    print(response)
    print("-" * 40)

Завантаження моделі Transformer (distilGPT2)...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 387.97it/s]



--- Генерація відповідей Transformer ---
Customer: I forgot my password.
Support:
If you are using a password that has been deleted, please refer to the following forum thread:
----------------------------------------
Customer: My order is delayed.
Support: Thank you for your support.
----------------------------------------
